In [4]:
from dotenv import dotenv_values
from pathlib import Path

BATCH_ID = BASE_DIR = INPUT_DIR = OUTPUT_DIR = ""

config = dotenv_values(".env")
for key, value in config.items():
    if key.endswith("_DIR"):
        globals()[key] = Path(value)
    else:
        globals()[key] = value

In [5]:
from PIL import Image
import pandas as pd

def scan_images(directory):

    records = []

    for path in sorted(directory.iterdir()):

        if path.suffix.lower() not in [".jpg", ".jpeg"]:
            continue

        try:
            with Image.open(path) as img:

                records.append({
                    "filename": path.name,
                    "filepath": str(path),
                    "width": img.width,
                    "height": img.height,
                    "megapixels": round(
                        img.width * img.height / 1_000_000, 2
                    ),
                    "format": img.format,
                    "size_mb": round(
                        path.stat().st_size / 1_000_000, 2
                    )
                })

        except Exception as e:
            print(f"Error: {path.name}: {e}")

    return pd.DataFrame(records)


df_images = scan_images(INPUT_DIR)

display(df_images)
print("Total images:", len(df_images))

,filename,filepath,width,height,megapixels,format,size_mb
0,DSCF0438.JPG,/home/fhri/Projects/shutterstock_metadata_gene...,6000,4000,24.0,JPEG,14.19
1,DSCF0556.JPG,/home/fhri/Projects/shutterstock_metadata_gene...,6000,4000,24.0,JPEG,12.42


Total images: 2


In [6]:
import ollama

response = ollama.list()
print("Available models:")
for model in response['models']:
    print(f"  - {model['model']}")
    MODEL = model['model']

Available models:
  - qwen2.5vl:3b


In [28]:
CATEGORY_GUIDELINES = """
OFFICIAL SHUTTERSTOCK CATEGORIES:

1. Abstract:
   Fractals, blur, patterns, textures and abstract concepts.

2. Animals/Wildlife:
   Domestic and wild animals.

3. Arts:
   Art objects, artwork, paintings and people making art.

4. Backgrounds/Textures:
   Images intended as backgrounds, textures, patterns,
   architectural details and flat lays.

5. Beauty/Fashion:
   Beauty products, hairstyles, clothing and fashion.

6. Buildings/Landmarks:
   Architectural structures, structural details
   and built environments.

7. Business/Finance:
   Businesspeople, workplaces, offices and financial subjects.

8. Celebrities:
   Celebrities and public figures.

9. Education:
   Schools, graduations and educational materials.

10. Food and Drink:
    Food, beverages, cooking and eating.

11. Healthcare/Medical:
    Health, wellness and medical subjects.

12. Holidays:
    Calendar events, celebrations, seasonal events and vacations.

13. Industrial:
    Construction, mining, electricity, tools,
    building materials and trades.

14. Interiors:
    Interior spaces of buildings and houses.

15. Miscellaneous:
    Subjects that do not fit another category.

16. Nature:
    Plants, animals, clouds, water and environmental subjects.

17. Objects:
    An object or group of objects as the main focus.

18. Parks/Outdoor:
    Parks, outdoor activities and camping.

19. People:
    Individuals, groups of people and body parts.

20. Religion:
    Religious and spiritual subjects.

21. Science:
    Scientific subjects, chemicals, living things and planets.

22. Signs/Symbols:
    Signs, flags, logos, icons and symbols.

23. Sports/Recreation:
    Sports, fitness and recreational hobbies.

24. Technology:
    Computers, phones, electronics and digital technology.

25. Transportation:
    Vehicles, roads, railways, aircraft and boats.

STRICT RULES:
- Select exactly 1 or 2 categories.
- Use ONLY the exact category names above.
- Never invent category names.
- Choose based on the actual visual subject.
- Do not select categories based on speculative context.
- Prefer the most specific relevant category.
"""

In [29]:
prompt = """
You are a professional stock photography
metadata specialist.

Analyze the provided photograph.

Generate Shutterstock metadata in JSON.

{CATEGORY_GUIDELINES}

Required fields:
- filename
- description
- keywords
- categories
- illustration
- mature_content
- editorial
- review_required
- review_reasons

Rules:
- Description must be in English.
- Generate 15-30 relevant keywords.
- Minimum 7 and maximum 50 keywords.
- No duplicate keywords.
- Use factual descriptions.
- Do not invent geographical information.
- Suggest valid Shutterstock categories based on CATEGORY_GUIDELINES.
- Flag visible logos and recognizable people.
- Do not make final legal eligibility decisions.
- Human approval is mandatory.

IMPORTANT:
Your task is to describe visual evidence,
NOT invent a story about the photograph.

STEP 1 - VISUAL IDENTIFICATION

Identify:
- Main physical object
- Secondary objects
- Visible people
- Environment
- Materials and textures
- Composition and depth of field

STEP 2 - CONTEXT CONTROL

Strict rules:
- Never infer occupation, identity, or legal status.
- Never assume a prison, hospital, factory, or other specific facility without visual evidence.
- Do not interpret people as prisoners, workers, customers, or criminals without evidence.
- Distinguish visible objects from inferred context.
- If uncertain, use a neutral physical description.

STEP 3 - STOCK METADATA

Generate:
- One factual description
- 20 relevant keywords
- Maximum 2 valid Shutterstock categories

Do not use keywords based on speculation.

Return valid JSON only.
"""

In [ ]:
import ollama
import base64
from PIL import Image
from io import BytesIO

def analyze_image_resized(image_path: str, prompt: str, max_size: int) -> str:
    """Resize image before sending to reduce token count."""
    
    # Open and resize image
    with Image.open(image_path) as img:
        # Convert to RGB if needed
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Resize maintaining aspect ratio
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        
        # Save to bytes
        buffer = BytesIO()
        img.save(buffer, format='JPEG', quality=100)
        image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    
    # Send to Ollama
    response = ollama.chat(
        model='qwen2.5vl:3b',
        messages = [{
            'role': 'user',
            'content': prompt,
            'images': [image_base64]
        }],
        options = {
            "temperature": 0,
            "seed": 42,
            "num_predict": 512
        }
    )
    
    return response['message']['content']

# Example usage
image_path = "/home/fhri/Projects/shutterstock_metadata_generator/shutterstock/input/batch_20261119/DSCF0438.JPG"

result = analyze_image_resized(image_path, prompt, max_size=1024)
print(result)

```json
{
  "filename": "metal_gates_with_people_out_of_focus.jpg",
  "description": "A close-up of a metal gate with a blurred background showing two people standing outside, possibly in a public area or a commercial zone.",
  "keywords": [
    "metal gate",
    "iron fence",
    "public area",
    "blurred background",
    "people",
    "out of focus",
    "public space",
    "indoor",
    "industrial",
    "metalwork",
    "fence",
    "security",
    "public building",
    "public entrance",
    "indoor gate",
    "indoor fence",
    "indoor security",
    "indoor gate",
    "indoor fence",
    "indoor security",
    "indoor gate",
    "indoor fence",
   
